# Part 2 — Bayesian Probability: Sentiment Keywords in IMDb Movie Reviews

**Question:** If a review contains a given keyword, what is the probability that the review is positive?

**Dataset:** Kaggle — *IMDB Dataset of 50K Movie Reviews* (`review`, `sentiment`). Stored compressed in `data/IMDB_Dataset.csv.gz` so it fits on GitHub.

**Basic Python only:** the standard-library modules `csv`, `gzip` and `string`. No pandas, NumPy, scikit-learn or NLTK.

**Our choice:** we compute **P(Positive | keyword)** for every keyword (we do not compute both directions). For a negative keyword, a *low* value is the signal; P(Negative | keyword) = 1 − P(Positive | keyword).

$$P(\text{Positive}\mid kw)=\frac{P(kw\mid\text{Positive})\;P(\text{Positive})}{P(kw)}$$

| Probability | Name | How we count it |
|---|---|---|
| P(Positive) | Prior | positive reviews / all reviews |
| P(kw \| Positive) | Likelihood | positive reviews containing kw / positive reviews |
| P(kw) | Marginal | reviews containing kw / all reviews |
| P(Positive \| kw) | Posterior | Bayes' theorem (above) |

In [1]:
import csv
import gzip
import string

DATA_PATH = "../data/IMDB_Dataset.csv.gz"
POSITIVE_KEYWORDS = ["excellent", "wonderful", "superb"]
NEGATIVE_KEYWORDS = ["worst", "waste", "awful"]
REJECTED_KEYWORDS = ["great", "bad", "masterpiece", "boring"]   # tested, then rejected (see section 4)

## 1. When does a review "contain" a keyword?

Lower-case the text, replace the HTML line breaks `<br />` found in this dataset, turn punctuation and digits into spaces, split into words, and keep the **set** of words. A review counts **once** even if it repeats the word. Example below.

In [2]:
CLEANER = str.maketrans({ch: " " for ch in string.punctuation + string.digits})


def tokenize(review):
    text = review.lower().replace("<br />", " ")
    return set(text.translate(CLEANER).split())


print(sorted(tokenize("Excellent!! An EXCELLENT film.<br /><br />10/10, not a waste of time.")))

['a', 'an', 'excellent', 'film', 'not', 'of', 'time', 'waste']


## 2. Counting — one pass over the 50,000 reviews

In [3]:
def count_reviews(path, keywords):
    counts = {"total": 0, "positive": 0}
    with_kw = {kw: 0 for kw in keywords}          # reviews containing kw
    with_kw_pos = {kw: 0 for kw in keywords}      # POSITIVE reviews containing kw
    with gzip.open(path, "rt", encoding="utf-8", newline="") as handle:
        for row in csv.DictReader(handle):
            is_positive = row["sentiment"] == "positive"
            words = tokenize(row["review"])
            counts["total"] += 1
            counts["positive"] += is_positive
            for kw in keywords:
                if kw in words:
                    with_kw[kw] += 1
                    with_kw_pos[kw] += is_positive
    return counts, with_kw, with_kw_pos


ALL_KEYWORDS = POSITIVE_KEYWORDS + NEGATIVE_KEYWORDS + REJECTED_KEYWORDS
counts, with_kw, with_kw_pos = count_reviews(DATA_PATH, ALL_KEYWORDS)
print(f"Reviews: {counts['total']}  positive: {counts['positive']}  negative: {counts['total'] - counts['positive']}")

Reviews: 50000  positive: 25000  negative: 25000


## 3. Bayes' theorem for each keyword

In [4]:
def bayes_posterior(likelihood, prior, marginal):
    # P(A|B) = P(B|A) * P(A) / P(B)
    return likelihood * prior / marginal


def keyword_probabilities(kw):
    prior = counts["positive"] / counts["total"]
    likelihood = with_kw_pos[kw] / counts["positive"]
    marginal = with_kw[kw] / counts["total"]
    return {"keyword": kw, "prior": prior, "likelihood": likelihood, "marginal": marginal,
            "posterior": bayes_posterior(likelihood, prior, marginal)}


def show_keyword(kw):
    r = keyword_probabilities(kw)
    n, n_pos, k, k_pos = counts["total"], counts["positive"], with_kw[kw], with_kw_pos[kw]
    print(f"Keyword '{kw}': in {k} reviews ({k_pos} positive, {k - k_pos} negative)")
    print(f"  Prior       P(Positive)          = {n_pos} / {n} = {r['prior']:.4f}")
    print(f"  Likelihood  P({kw}|Positive) = {k_pos} / {n_pos} = {r['likelihood']:.4f}")
    print(f"  Marginal    P({kw})          = {k} / {n} = {r['marginal']:.4f}")
    print(f"  Posterior   P(Positive|{kw}) = {r['likelihood']:.6f} x {r['prior']:.4f} / {r['marginal']:.6f}"
          f" = {r['posterior']:.4f}")
    print(f"  Check by direct counting: {k_pos} / {k} = {k_pos / k:.4f}\n")


for kw in POSITIVE_KEYWORDS + NEGATIVE_KEYWORDS:
    show_keyword(kw)

Keyword 'excellent': in 3552 reviews (2868 positive, 684 negative)
  Prior       P(Positive)          = 25000 / 50000 = 0.5000
  Likelihood  P(excellent|Positive) = 2868 / 25000 = 0.1147
  Marginal    P(excellent)          = 3552 / 50000 = 0.0710
  Posterior   P(Positive|excellent) = 0.114720 x 0.5000 / 0.071040 = 0.8074
  Check by direct counting: 2868 / 3552 = 0.8074

Keyword 'wonderful': in 2780 reviews (2258 positive, 522 negative)
  Prior       P(Positive)          = 25000 / 50000 = 0.5000
  Likelihood  P(wonderful|Positive) = 2258 / 25000 = 0.0903
  Marginal    P(wonderful)          = 2780 / 50000 = 0.0556
  Posterior   P(Positive|wonderful) = 0.090320 x 0.5000 / 0.055600 = 0.8122
  Check by direct counting: 2258 / 2780 = 0.8122

Keyword 'superb': in 1199 reviews (1026 positive, 173 negative)
  Prior       P(Positive)          = 25000 / 50000 = 0.5000
  Likelihood  P(superb|Positive) = 1026 / 25000 = 0.0410
  Marginal    P(superb)          = 1199 / 50000 = 0.0240
  Posterior   P(

In [5]:
from IPython.display import Markdown, display   # only to display the tables nicely


def markdown_table(keywords, title):
    lines = [f"**{title}**", "",
             "| Keyword | Prior P(Positive) | Likelihood P(kw\\|Positive) | Marginal P(kw) | Posterior P(Positive\\|kw) |",
             "|---|---|---|---|---|"]
    for kw in keywords:
        r = keyword_probabilities(kw)
        lines.append(f"| {kw} | {r['prior']:.4f} | {r['likelihood']:.4f} | {r['marginal']:.4f} | **{r['posterior']:.4f}** |")
    display(Markdown("\n".join(lines)))


markdown_table(POSITIVE_KEYWORDS, "Positive keywords")
markdown_table(NEGATIVE_KEYWORDS, "Negative keywords")
markdown_table(REJECTED_KEYWORDS, "Rejected candidates")

**Positive keywords**

| Keyword | Prior P(Positive) | Likelihood P(kw\|Positive) | Marginal P(kw) | Posterior P(Positive\|kw) |
|---|---|---|---|---|
| excellent | 0.5000 | 0.1147 | 0.0710 | **0.8074** |
| wonderful | 0.5000 | 0.0903 | 0.0556 | **0.8122** |
| superb | 0.5000 | 0.0410 | 0.0240 | **0.8557** |

**Negative keywords**

| Keyword | Prior P(Positive) | Likelihood P(kw\|Positive) | Marginal P(kw) | Posterior P(Positive\|kw) |
|---|---|---|---|---|
| worst | 0.5000 | 0.0164 | 0.0887 | **0.0927** |
| waste | 0.5000 | 0.0070 | 0.0507 | **0.0691** |
| awful | 0.5000 | 0.0114 | 0.0576 | **0.0985** |

**Rejected candidates**

| Keyword | Prior P(Positive) | Likelihood P(kw\|Positive) | Marginal P(kw) | Posterior P(Positive\|kw) |
|---|---|---|---|---|
| great | 0.5000 | 0.3391 | 0.2515 | **0.6741** |
| bad | 0.5000 | 0.1184 | 0.2357 | **0.2511** |
| masterpiece | 0.5000 | 0.0351 | 0.0241 | **0.7274** |
| boring | 0.5000 | 0.0237 | 0.0610 | **0.1940** |

## 4. Why these keywords (justified with our counts)

**Positive — excellent (0.8074), wonderful (0.8122), superb (0.8557).** Each raises the probability of a positive review from the 0.50 prior to above 0.80, and each is common enough to be reliable (3,552 / 2,780 / 1,199 reviews).

**Negative — worst (0.0927), waste (0.0691), awful (0.0985).** Each drops P(Positive) below 0.10, so more than 90% of reviews with these words are negative; found in 4,434 / 2,534 / 2,882 reviews. "waste" mostly appears as "waste of time/money" (in 1,212 negative reviews vs 57 positive).

**Rejected after testing:**
* *great* (0.6741) — appears in 4,099 negative reviews: too common in both classes.
* *bad* (0.2511) — appears in 2,960 positive reviews; e.g. "bad guy" occurs in 382 positive reviews.
* *masterpiece* (0.7274) — 329 negative reviews use it for *another* film ("the original was a masterpiece") or sarcastically.
* *boring* (0.1940) — useful but weaker than our three; "never boring" appears in 29 positive reviews.

**Note:** the prior is exactly 0.5 because the dataset is balanced (25,000 / 25,000). That is why the posterior equals the direct count (positive reviews with the word ÷ reviews with the word) — the "check" line above confirms it for every keyword.